# Pipeline final para entrega

Este notebook toma los datos raw de Higgs (`training.csv` y `test.csv`), reconstruye el preprocesamiento `P04`, entrena el modelo final y genera el CSV de submission correspondiente al mejor public score obtenido.

Modelo final: `P04 + XGBoost SW04 + CV ensemble 5 folds + top 15.75% como signal`.

In [ ]:
from pathlib import Path
import sys

stage_dir = Path.cwd()
if not (stage_dir / "final_public_pipeline.py").exists():
    stage_dir = Path.cwd() / "100. Pipeline"
sys.path.insert(0, str(stage_dir.resolve()))

from final_public_pipeline import (
    CONFIG_ID,
    FINAL_XGB_PARAMS,
    MODEL_NAME,
    N_SPLITS,
    PIPELINE_ID,
    REFERENCE_KAGGLE,
    SEED,
    TOP_SIGNAL_PERCENT,
    find_project_root,
    run_final_public_pipeline,
)

In [ ]:
project_root = find_project_root()

config = {
    "raw_train_path": str(project_root / "data" / "higgs-boson" / "training.csv"),
    "raw_test_path": str(project_root / "data" / "higgs-boson" / "test.csv"),
    "pipeline": PIPELINE_ID,
    "model": MODEL_NAME,
    "config_id": CONFIG_ID,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "sample_weight_used": True,
    "weight_used_as_feature": False,
    "threshold_policy": f"top_{TOP_SIGNAL_PERCENT:g}_percent_by_score_rank",
    "reference_kaggle": REFERENCE_KAGGLE,
    "hyperparameters": FINAL_XGB_PARAMS,
}
config

In [ ]:
outputs = run_final_public_pipeline(project_root=project_root)

print(f"scores_path: {outputs['scores_path']}")
print(f"submission_path: {outputs['submission_path']}")
print(f"metadata_path: {outputs['metadata_path']}")
outputs["metadata"]["validation"]

In [ ]:
outputs["metadata"]["processed_shapes"]

In [ ]:
display(outputs["scores"].head())
display(outputs["submission"].head())
outputs["submission"]["Class"].value_counts(normalize=True).rename("rate") * 100